In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
import pandas as pd
import numpy as np

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

id2label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

deberta_model_name = "microsoft/deberta-v3-small"
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_model_name)
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    deberta_model_name,
    num_labels=5,
    id2label=id2label,
    label2id=label2id
).to(device)

roberta_model_name = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta_model_name,
    num_labels=5,
    id2label=id2label,
    label2id=label2id
).to(device)

print("Models and tokenizers loaded successfully.")

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Models and tokenizers loaded successfully.


In [5]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [6]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
idx_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

row_25 = train_df.iloc[25]
prompt_text = str(row_25['prompt'])
options_text = " ".join([f"{opt}: {row_25[opt]}" for opt in ['A', 'B', 'C', 'D', 'E']])
input_text = f"{prompt_text} {options_text}"

deberta_model_name = "microsoft/deberta-v3-small"
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_model_name)
deberta_model = AutoModelForSequenceClassification.from_pretrained(deberta_model_name, num_labels=5).to(device)

inputs = deberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)

deberta_model.eval()
with torch.no_grad():
    outputs = deberta_model(**inputs)
    probs = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()

top_idx = probs.argmax()
top_option = idx_to_letter[top_idx]
top_prob = probs[top_idx]

print(f"{top_option}, {top_prob:.3f}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight       

B, 0.225


In [7]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
idx_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

row_25 = train_df.iloc[25]
prompt_text = str(row_25['prompt'])
options_text = " ".join([f"{opt}: {row_25[opt]}" for opt in ['A', 'B', 'C', 'D', 'E']])
input_text = f"{prompt_text} {options_text}"

deberta_model_name = "microsoft/deberta-v3-small"
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_model_name)
deberta_model = AutoModelForSequenceClassification.from_pretrained(deberta_model_name, num_labels=5).to(device)

roberta_model_name = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_model_name, num_labels=5).to(device)

deberta_inputs = deberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
roberta_inputs = roberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)

deberta_model.eval()
roberta_model.eval()

with torch.no_grad():
    deberta_probs = F.softmax(deberta_model(**deberta_inputs).logits, dim=-1).squeeze().cpu().numpy()
    roberta_probs = F.softmax(roberta_model(**roberta_inputs).logits, dim=-1).squeeze().cpu().numpy()

avg_probs = (deberta_probs + roberta_probs) / 2.0

top_idx = avg_probs.argmax()
top_option = idx_to_letter[top_idx]

print(top_option)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight       

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


A


In [8]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
idx_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

row_25 = train_df.iloc[25]
prompt_text = str(row_25['prompt'])
options_text = " ".join([f"{opt}: {row_25[opt]}" for opt in ['A', 'B', 'C', 'D', 'E']])
input_text = f"{prompt_text} {options_text}"

deberta_model_name = "microsoft/deberta-v3-small"
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_model_name)
deberta_model = AutoModelForSequenceClassification.from_pretrained(deberta_model_name, num_labels=5).to(device)

roberta_model_name = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_model_name, num_labels=5).to(device)

deberta_inputs = deberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
roberta_inputs = roberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)

deberta_model.eval()
roberta_model.eval()

with torch.no_grad():
    deberta_probs = F.softmax(deberta_model(**deberta_inputs).logits, dim=-1).squeeze().cpu().numpy()
    roberta_probs = F.softmax(roberta_model(**roberta_inputs).logits, dim=-1).squeeze().cpu().numpy()

weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

top_idx = weighted_probs.argmax()
top_option = idx_to_letter[top_idx]

print(top_option)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight       

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


A


In [9]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
idx_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

row_25 = train_df.iloc[25]
prompt_text = str(row_25['prompt'])
options_text = " ".join([f"{opt}: {row_25[opt]}" for opt in ['A', 'B', 'C', 'D', 'E']])
input_text = f"{prompt_text} {options_text}"

deberta_model_name = "microsoft/deberta-v3-small"
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_model_name)
deberta_model = AutoModelForSequenceClassification.from_pretrained(deberta_model_name, num_labels=5).to(device)

roberta_model_name = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_model_name, num_labels=5).to(device)

deberta_inputs = deberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
roberta_inputs = roberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)

deberta_model.eval()
roberta_model.eval()

with torch.no_grad():
    deberta_probs = F.softmax(deberta_model(**deberta_inputs).logits, dim=-1).squeeze().cpu().numpy()
    roberta_probs = F.softmax(roberta_model(**roberta_inputs).logits, dim=-1).squeeze().cpu().numpy()

weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

ranked_indices = weighted_probs.argsort()[::-1]
top_3_letters = [idx_to_letter[i] for i in ranked_indices[:3]]

prediction_string = " ".join(top_3_letters)
print(prediction_string)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight       

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


B E C


In [10]:
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
idx_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

deberta_model_name = "microsoft/deberta-v3-small"
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_model_name)
deberta_model = AutoModelForSequenceClassification.from_pretrained(deberta_model_name, num_labels=5).to(device)

roberta_model_name = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_model_name, num_labels=5).to(device)

deberta_model.eval()
roberta_model.eval()

predictions = []

with torch.no_grad():
    for _, row in test_df.iterrows():
        prompt_text = str(row['prompt'])
        options_text = " ".join([f"{opt}: {row[opt]}" for opt in ['A', 'B', 'C', 'D', 'E']])
        input_text = f"{prompt_text} {options_text}"
        
        deberta_inputs = deberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
        roberta_inputs = roberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
        
        deberta_probs = F.softmax(deberta_model(**deberta_inputs).logits, dim=-1).squeeze().cpu().numpy()
        roberta_probs = F.softmax(roberta_model(**roberta_inputs).logits, dim=-1).squeeze().cpu().numpy()
        
        weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)
        
        ranked_indices = weighted_probs.argsort()[::-1]
        top_3_letters = [idx_to_letter[i] for i in ranked_indices[:3]]
        
        predictions.append(" ".join(top_3_letters))

submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': predictions
})

submission_df.to_csv('submission.csv', index=False)

print(len(submission_df))

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight       

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


500


In [11]:
tta_changes = 0

with torch.no_grad():
    for i in range(50):
        row = test_df.iloc[i]
        prompt_text = str(row['prompt'])
        options_text = " ".join([f"{opt}: {row[opt]}" for opt in ['A', 'B', 'C', 'D', 'E']])
        
        # 1. Original
        input_orig = f"{prompt_text} {options_text}"
        inputs_orig = deberta_tokenizer(input_orig, return_tensors="pt", truncation=True, max_length=512).to(device)
        probs_orig = F.softmax(deberta_model(**inputs_orig).logits, dim=-1).squeeze().cpu().numpy()
        
        # 2. Instruction-augmented
        input_aug = f"Answer the following multiple-choice question carefully: {prompt_text} {options_text}"
        inputs_aug = deberta_tokenizer(input_aug, return_tensors="pt", truncation=True, max_length=512).to(device)
        probs_aug = F.softmax(deberta_model(**inputs_aug).logits, dim=-1).squeeze().cpu().numpy()
        
        # Averaged
        probs_tta = (probs_orig + probs_aug) / 2.0
        
        if probs_orig.argmax() != probs_tta.argmax():
            tta_changes += 1

print(tta_changes)

9


In [13]:
different_top1_count = 0

with torch.no_grad():
    for i in range(100):
        row = test_df.iloc[i]
        prompt_text = str(row['prompt'])
        options_text = " ".join([f"{opt}: {row[opt]}" for opt in ['A', 'B', 'C', 'D', 'E']])
        input_text = f"{prompt_text} {options_text}"
        
        deberta_inputs = deberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
        roberta_inputs = roberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
        
        deberta_probs = F.softmax(deberta_model(**deberta_inputs).logits, dim=-1).squeeze().cpu().numpy()
        roberta_probs = F.softmax(roberta_model(**roberta_inputs).logits, dim=-1).squeeze().cpu().numpy()
        
        weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)
        
        deberta_top1 = deberta_probs.argmax()
        weighted_top1 = weighted_probs.argmax()
        
        if deberta_top1 != weighted_top1:
            different_top1_count += 1

print(different_top1_count)

4


In [14]:
positive_gain_count = 0

with torch.no_grad():
    for i in range(100):
        row = test_df.iloc[i]
        prompt_text = str(row['prompt'])
        options_text = " ".join([f"{opt}: {row[opt]}" for opt in ['A', 'B', 'C', 'D', 'E']])
        input_text = f"{prompt_text} {options_text}"
        
        deberta_inputs = deberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
        roberta_inputs = roberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
        
        deberta_probs = F.softmax(deberta_model(**deberta_inputs).logits, dim=-1).squeeze().cpu().numpy()
        roberta_probs = F.softmax(roberta_model(**roberta_inputs).logits, dim=-1).squeeze().cpu().numpy()
        
        weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)
        
        deberta_conf = deberta_probs.max()
        ensemble_conf = weighted_probs.max()
        
        confidence_gain = ensemble_conf - deberta_conf
        
        if confidence_gain > 0:
            positive_gain_count += 1

print(positive_gain_count)

0


In [15]:
top3_changed_count = 0

with torch.no_grad():
    for i in range(100):
        row = test_df.iloc[i]
        prompt_text = str(row['prompt'])
        options_text = " ".join([f"{opt}: {row[opt]}" for opt in ['A', 'B', 'C', 'D', 'E']])
        input_text = f"{prompt_text} {options_text}"
        
        deberta_inputs = deberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
        roberta_inputs = roberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
        
        deberta_probs = F.softmax(deberta_model(**deberta_inputs).logits, dim=-1).squeeze().cpu().numpy()
        roberta_probs = F.softmax(roberta_model(**roberta_inputs).logits, dim=-1).squeeze().cpu().numpy()
        
        weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)
        
        deberta_top3 = [idx_to_letter[idx] for idx in deberta_probs.argsort()[::-1][:3]]
        ensemble_top3 = [idx_to_letter[idx] for idx in weighted_probs.argsort()[::-1][:3]]
        
        if deberta_top3 != ensemble_top3:
            top3_changed_count += 1

print(top3_changed_count)

14


In [19]:
def map_at_3(actual, predicted):
    scores = []
    for true_ans, pred_list in zip(actual, predicted):
        score = 0
        for rank, pred in enumerate(pred_list[:3], start=1):
            if pred == true_ans:
                score = 1 / rank
                break
        scores.append(score)
    return np.mean(scores)

final_map3 = map_at_3(actual_labels, predicted_top3)
print(f"{final_map3:.4f}")

0.4317


In [20]:
actual_labels = []
predicted_top3 = []

with torch.no_grad():
    for i in range(100):
        row = train_df.iloc[i]
        prompt_text = str(row['prompt'])
        options_text = " ".join([f"{opt}: {row[opt]}" for opt in ['A', 'B', 'C', 'D', 'E']])
        input_text = f"{prompt_text} {options_text}"
        
        deberta_inputs = deberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
        roberta_inputs = roberta_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
        
        deberta_probs = F.softmax(deberta_model(**deberta_inputs).logits, dim=-1).squeeze().cpu().numpy()
        roberta_probs = F.softmax(roberta_model(**roberta_inputs).logits, dim=-1).squeeze().cpu().numpy()
        
        weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)
        
        ranked_indices = weighted_probs.argsort()[::-1][:3]
        top_3 = [idx_to_letter[idx] for idx in ranked_indices]
        
        predicted_top3.append(top_3)
        actual_labels.append(row['answer'])

final_map3 = map_at_3(actual_labels, predicted_top3)
print(f"{final_map3:.4f}")

0.4317
